# Generic Solver Rule Families

**Status:** Architecture approved; written specification awaiting review  
**Date:** 2026-08-20  
**Design epic:** `bd-3he`

This notebook is the authoritative design for adding three generic, catalog-visible solver families to `spur-solver`: configuration/compatibility, scheduling/allocation, and workflow/state-transition.

Live NS-Mermaid profile pins:

- `relational_lia@1` — implemented; used for family routing, configuration composition, and bounded scheduling feasibility.
- `state_invariant_lia@1` — implemented; used for workflow inductive safety.
- `schedule_optimization@1` — capability unavailable; optimization evidence is provided by persisted typed solver runs.
- `transition_bmc@1` — capability unavailable; bounded reachability evidence is provided by persisted typed solver runs.

The notebook never treats feasibility as optimality and never treats finite-horizon UNSAT as an unbounded temporal proof.

## Decision and scope

Implement three distinct manifest-backed native families:

1. `configuration` with profile `finite_compatibility`.
2. `scheduling` with profile `finite_horizon`.
3. `workflow` with profile `bounded_trace`.

Each family owns a strict fact model and native lowering module. YAML owns stable IDs, descriptions, parameters, examples, and handler declarations; Rust owns reference resolution, bounded variable generation, checked arithmetic, typed constraint lowering, objectives, and result projection.

### Goals

- Make common compatibility rules discoverable through `solve_rule_spec`.
- Demonstrate real allocation optimization through `solve_rules` synthesis.
- Support observed-trace verification plus bounded workflow witness and counterexample generation.
- Preserve solver status, timeout, resource caps, rule attribution, and UNSAT-core behavior.

### Non-goals

- Runtime loading of user-authored manifests or raw SMT inside manifests.
- Automatic SemVer parsing or inference of compatibility from version precedence.
- Continuous time, preemption, sequence-dependent setup, or travel-time scheduling.
- Unbounded liveness, fairness, infinite-cycle/lasso checking, or dynamic workflow state creation.
- Extending NS-Mermaid itself with scheduling or BMC adapters.

## Integration architecture

The change extends the existing hybrid catalog rather than introducing a second solver path.

- Add `configuration.rs`, `scheduling.rs`, and `workflow.rs` family entry modules under `crates/spur-solver/src/rules/families/`.
- Add one subdirectory per family containing `family.yaml`, `compile.rs`, typed model helpers as needed, and one YAML rule manifest per seed rule.
- Register the three compilers in `families::compilers()`.
- Extend `NativeHandlerV1` with closed, family-owned handler variants and exact parameter ABIs.
- Continue deriving top-level family and executable rule schemas dynamically from validated manifests.
- Compile every family into the existing typed `SolveConstraintsRequest`; scheduling synthesis may additionally emit the existing typed `Objective::Minimize`.
- Keep verification projection and rule attribution in the current execution layer.

The routing contract below makes the new family discriminator explicit and rejects ambiguous or absent routing.

In [ ]:
flowchart TD
    SPEC["`@spec GENERIC-FAMILY-ROUTING
@type Status = enum[configuration, scheduling, workflow, invalid]
@input configuration_requested: Bool
@input scheduling_requested: Bool
@input workflow_requested: Bool
@output status: Status
@requires PRE: true`"]

    CONFIGURATION["`@branch CONFIGURATION
@when configuration_requested and not scheduling_requested and not workflow_requested
@ensures CONFIGURATION_STATUS: status = configuration`"]

    SCHEDULING["`@branch SCHEDULING
@when not configuration_requested and scheduling_requested and not workflow_requested
@ensures SCHEDULING_STATUS: status = scheduling`"]

    WORKFLOW["`@branch WORKFLOW
@when not configuration_requested and not scheduling_requested and workflow_requested
@ensures WORKFLOW_STATUS: status = workflow`"]

    INVALID["`@branch INVALID
@when not ((configuration_requested and not scheduling_requested and not workflow_requested) or (not configuration_requested and scheduling_requested and not workflow_requested) or (not configuration_requested and not scheduling_requested and workflow_requested))
@ensures INVALID_STATUS: status = invalid`"]

    CHECK["`@verify ROUTE_DETERMINISTIC: prove determinism
@verify ROUTE_COVERAGE: prove partition_coverage
@verify ROUTE_EXCLUSIVE: prove partition_exclusive
@verify ROUTE_STATUSES: witness each status`"]

    SPEC --> CONFIGURATION --> CHECK
    SPEC --> SCHEDULING --> CHECK
    SPEC --> WORKFLOW --> CHECK
    SPEC --> INVALID --> CHECK

## Configuration / compatibility

Let each component selection be `x_c ∈ {0,1}`. Enum and version domains are finite and declared in facts. Version strings are normalized by the caller into ordered integer ranks.

| Rule ID | Mathematical contract | Required configuration |
|---|---|---|
| `configuration.requires_any` | `¬x_c ∨ ⋁_{p∈P} x_p` | consumer reference and non-empty finite provider set |
| `configuration.excludes` | `¬x_a ∨ ¬x_b` | two resolved component references |
| `configuration.selection_cardinality` | `z_i ≤ g` and `L·g ≤ Σ_i z_i ≤ U·g` | group, members, `0 ≤ L ≤ U ≤ n` |
| `configuration.attribute_allowed_pair` | `¬active ∨ ⋁_{(u,v)∈A}(left=u ∧ right=v)` | two finite enum attributes and explicit allowed tuple set |
| `configuration.version_interval` | `¬x_c ∨ (x_p ∧ min≤rank_p≤max)` | consumer, provider, ordering ID, inclusive rank interval |

Rule configuration is rejected before solving when references are unresolved, IDs are duplicated, sets are empty where prohibited, bounds are inverted, enum labels are unknown, or version ordering is undeclared.

Synthesis only creates variables for fields explicitly declared unknown and only within declared finite domains. The solver must not invent components, providers, enum labels, or version ranks outside the fact model.

Local JSON/YAML shape, scalar bounds, and required fields stay in schema validation. Cross-entity dependencies, exclusions, global cardinality, attribute joins, version selection, and composition across multiple rules belong in SMT lowering.

The following formal cell instantiates all five rule shapes into one valid/invalid decision relation and proves that the aggregate result is deterministic, covering, exclusive, and admits both statuses.

In [ ]:
flowchart TD
    SPEC["`@spec CONFIGURATION-FINITE-COMPATIBILITY
@type Status = enum[valid, invalid]
@input consumer_selected: Bool
@input provider_a_selected: Bool
@input provider_b_selected: Bool
@input group_active: Bool
@input choice_count: Int
@input attribute_match: Bool
@input provider_version_rank: Int
@output status: Status
@requires CHOICE_DOMAIN: 0 <= choice_count and choice_count <= 2
@requires VERSION_DOMAIN: 0 <= provider_version_rank and provider_version_rank <= 6`"]

    VALID["`@branch VALID
@when (not consumer_selected or provider_a_selected or provider_b_selected) and (not provider_a_selected or not provider_b_selected) and ((not group_active and choice_count = 0) or (group_active and 1 <= choice_count and choice_count <= 2)) and (not consumer_selected or attribute_match) and (not consumer_selected or (provider_a_selected and 2 <= provider_version_rank and provider_version_rank <= 4))
@ensures VALID_STATUS: status = valid`"]

    INVALID["`@branch INVALID
@when not ((not consumer_selected or provider_a_selected or provider_b_selected) and (not provider_a_selected or not provider_b_selected) and ((not group_active and choice_count = 0) or (group_active and 1 <= choice_count and choice_count <= 2)) and (not consumer_selected or attribute_match) and (not consumer_selected or (provider_a_selected and 2 <= provider_version_rank and provider_version_rank <= 4)))
@ensures INVALID_STATUS: status = invalid`"]

    CHECK["`@verify CONFIG_DETERMINISTIC: prove determinism
@verify CONFIG_COVERAGE: prove partition_coverage
@verify CONFIG_EXCLUSIVE: prove partition_exclusive
@verify CONFIG_STATUSES: witness each status`"]

    SPEC --> VALID --> CHECK
    SPEC --> INVALID --> CHECK

## Scheduling / allocation

Facts declare finite jobs `J`, machines `M`, resources `R`, and integer horizon `H > 0`. For each valid placement, the compiler creates `x_jmt ∈ {0,1}`, meaning job `j` starts on machine `m` at tick `t`.

`D_j = {(m,t) | 0 ≤ t and t + p_jm ≤ H}`

`S_j = Σ_(m,t∈D_j) t·x_jmt` and `C_j = Σ_(m,t∈D_j) (t+p_jm)·x_jmt`.

| Rule ID | Mathematical contract |
|---|---|
| `scheduling.assignment_exactly_once` | `∀j: Σ_(m,t∈D_j) x_jmt = 1` |
| `scheduling.placement_allowed` | `x_jmt = 0` for ineligible machines or placements outside release/deadline windows |
| `scheduling.precedence_finish_start` | `∀(i,j,ℓ): C_i + ℓ ≤ S_j` |
| `scheduling.cumulative_capacity` | for every `(m,r,τ)`, active demand `Σ q_jr·x_jmt ≤ b_mr` |
| `scheduling.minimize_makespan` | `Cmax ≥ C_j` for every job; synthesize with objective `minimize Cmax` |

`minimize_makespan` has mode-sensitive behavior: synthesis emits the objective; verification requires `maximum_makespan` and emits the hard bound `Cmax ≤ maximum_makespan`. A verification request without that bound is rejected as meaningless.

Generated variables scale approximately with `Σ_j,m (H - p_jm + 1)` and capacity constraints with `|M|·|R|·H`. Compilation therefore enforces checked arithmetic plus explicit horizon, placement-variable, and generated-constraint limits before launching Z3.

The following formal cell verifies a representative hard-feasibility partition. NS-Mermaid does not currently support optimization, so exact optimality remains bound to typed solver result `sol_cdfeef6018084ce6` (complete optimum `Cmax=4`) and lower-bound result `sol_18ead23e6168487f` (`Cmax≤3` is UNSAT).

In [ ]:
flowchart TD
    SPEC["`@spec SCHEDULING-FINITE-HORIZON-FEASIBILITY
@type Machine = enum[m1, m2]
@type Status = enum[feasible, infeasible]
@input start_a: Int
@input start_b: Int
@input machine_a: Machine
@input machine_b: Machine
@input eligible_a: Bool
@input eligible_b: Bool
@input precedence_ab: Bool
@output status: Status
@requires START_DOMAIN: 0 <= start_a and start_a + 2 <= 4 and 0 <= start_b and start_b + 2 <= 4`"]

    FEASIBLE["`@branch FEASIBLE
@when eligible_a and eligible_b and (not precedence_ab or start_a + 2 <= start_b) and (machine_a != machine_b or start_a + 2 <= start_b or start_b + 2 <= start_a)
@ensures FEASIBLE_STATUS: status = feasible`"]

    INFEASIBLE["`@branch INFEASIBLE
@when not (eligible_a and eligible_b and (not precedence_ab or start_a + 2 <= start_b) and (machine_a != machine_b or start_a + 2 <= start_b or start_b + 2 <= start_a))
@ensures INFEASIBLE_STATUS: status = infeasible`"]

    CHECK["`@verify SCHEDULE_DETERMINISTIC: prove determinism
@verify SCHEDULE_COVERAGE: prove partition_coverage
@verify SCHEDULE_EXCLUSIVE: prove partition_exclusive
@verify SCHEDULE_STATUSES: witness each status`"]

    SPEC --> FEASIBLE --> CHECK
    SPEC --> INFEASIBLE --> CHECK

## Workflow / state transition

Normalize each subject as a finite trace:

`τ = (q_0, a_0, q_1, …, a_(L-1), q_L), 0 ≤ L ≤ K`

Facts declare finite state enum `Q`, event enum `A`, allowed initial set `I`, per-step enabled relation `R_i ⊆ Q×A×Q`, safe set `S`, and target set `G`. Callers flatten hierarchical/parallel configurations and resolve guards and transition priority into `R_i` before solving.

| Rule ID | Mathematical contract |
|---|---|
| `workflow.initial_state_allowed` | `q_0 ∈ I` |
| `workflow.transition_allowed` | `⋀_(i=0..L-1) (q_i,a_i,q_(i+1)) ∈ R_i` |
| `workflow.safety_invariant` | `⋀_(i=0..L) q_i ∈ S` |
| `workflow.bounded_reachability` | `⋁_(i=0..min(b,L)) q_i ∈ G` |

Observed-trace verification checks the first three rules directly. Counterexample search combines initial-state and transition validity with bounded reachability where `G` is the unsafe-state set: SAT returns a concrete violating trace; UNSAT proves only that no violating trace exists within the declared finite domains and bound.

Termination or stuttering must be represented explicitly, otherwise exact-horizon analysis can omit shorter deadlocked traces. Reachability synthesis proves existence of one bounded path, not eventual success on every execution.

The following state cell proves the inductive invariant `approved ⇒ reviewed` for initialization and every declared approval-workflow transition. Native bounded-trace proof is unavailable in NS-Mermaid, so concrete trace evidence remains bound to `sol_5451a6ecf5944730` and `sol_b7f9555965a443cc`.

In [ ]:
stateDiagram-v2
    [*] --> Draft
    Draft --> Review: submit
    Review --> Approved: approve
    Review --> Rejected: reject

    note right of Draft
      @spec WORKFLOW-APPROVAL-SAFETY
      @type WorkflowEvent = enum[submit, approve, reject]
      @input event: WorkflowEvent
      @state-var reviewed: Bool
      @state-var approved: Bool
      @requires INITIAL: reviewed = false and approved = false
      @state Draft
      @invariant APPROVAL_REQUIRES_REVIEW: not approved or reviewed
    end note

    note right of Review
      @state Review
      @transition SUBMIT
      @from Draft
      @to Review
      @event event = submit
      @guard reviewed = false and approved = false
      @update reviewed' = true
      @update approved' = false
    end note

    note right of Approved
      @state Approved
      @transition APPROVE
      @from Review
      @to Approved
      @event event = approve
      @guard reviewed = true and approved = false
      @update reviewed' = true
      @update approved' = true
    end note

    note right of Rejected
      @state Rejected
      @transition REJECT
      @from Review
      @to Rejected
      @event event = reject
      @guard reviewed = true and approved = false
      @update reviewed' = true
      @update approved' = false
      @verify WORKFLOW_INIT_SAFE: prove initiate APPROVAL_REQUIRES_REVIEW
      @verify WORKFLOW_SUBMIT_SAFE: prove preserve APPROVAL_REQUIRES_REVIEW on SUBMIT
      @verify WORKFLOW_APPROVE_SAFE: prove preserve APPROVAL_REQUIRES_REVIEW on APPROVE
      @verify WORKFLOW_REJECT_SAFE: prove preserve APPROVAL_REQUIRES_REVIEW on REJECT
    end note

## Manifest and compiler contracts

Each family manifest declares its stable family ID, profile, subject kind, facts schema, unknown-field schema, and supported handler set. Each rule manifest declares one stable rule ID, parameters schema, native handler, examples, and authority links.

Compiler obligations:

1. Validate manifest/handler ownership and exact parameter ABI during catalog initialization.
2. Validate facts and resolve every referenced ID before allocating solver variables.
3. Normalize unordered inputs deterministically so variable names and attribution are stable.
4. Enforce finite domains, model-size budgets, checked integer conversions, and limit errors before solver execution.
5. Attach stable constraint names carrying rule and subject identity.
6. Build only typed `ConstraintExpr` and `Objective` values.
7. Re-run generic request semantic validation before execution.
8. Preserve raw `sat`, `unsat`, `unknown`, and `timeout` statuses; never collapse incomplete optimization into proven optimality.

Invalid schemas, malformed rule parameters, unresolved references, and size-limit failures are compilation errors. Logical conflicts among otherwise valid rules are solver outcomes and retain UNSAT-core attribution.

## Verification strategy

Every seed rule receives dual evaluation:

- one valid verification vector;
- one invalid vector with expected rule/subject attribution;
- one bounded synthesis vector where the rule admits unknowns;
- one negated or infeasible counterexample query;
- manifest ABI and catalog-discovery coverage.

Family-level scenarios:

- Configuration: compose requires, exclusion, cardinality, attribute, and version rules; test both satisfiable selection and conflicting UNSAT core.
- Scheduling: test feasible allocation, complete makespan optimization, and an impossible lower bound.
- Workflow: test a valid observed trace, an illegal transition, a reachable unsafe trace, and an unreachable unsafe target within a fixed bound.

Persisted research evidence:

- Configuration witnesses: `sol_b7e4a480b2814610`, `sol_7ac2b843093d4668`, `sol_4512318a09604d47`, `sol_189d68a065a64176`, `sol_3d6326fb72b64c33`.
- Scheduling: `sol_9994a81fbabf4588`, `sol_cdfeef6018084ce6`, `sol_18ead23e6168487f`.
- Workflow: `sol_6a159b71b73f4d57`, `sol_5451a6ecf5944730`, `sol_b7f9555965a443cc`.

Repository verification uses `scripts/spur-cargo fmt --check`, focused `spur-solver` tests, and workspace-compatible checks. Implementation follows RED/GREEN commits and receives independent code review before completion.

## Research basis

Configuration semantics are grounded in Debian package relationships, Linux Kconfig dependencies/choices, OASIS TOSCA requirements and capability matching, FODA feature relations, and explicit version ordering.

Scheduling uses established zero-one time-indexed allocation, finish-to-start precedence, cumulative capacity, and makespan optimization as documented by Pritsker–Watters–Wolfe, OR-Tools, and MiniZinc.

Workflow semantics follow the TLA+ `Init ∧ Next` model, W3C SCXML transition semantics, and bounded counterexample formulations documented by Apalache and Alloy.

Primary sources:

- https://www.debian.org/doc/debian-policy/ch-relationships.html
- https://docs.kernel.org/kbuild/kconfig-language.html
- https://docs.oasis-open.org/tosca/TOSCA/v2.0/TOSCA-v2.0.html
- https://resources.sei.cmu.edu/asset_files/TechnicalReport/1990_005_001_15872.pdf
- https://pubsonline.informs.org/doi/10.1287/mnsc.16.1.93
- https://developers.google.com/optimization/scheduling/job_shop
- https://docs.minizinc.dev/en/2.9.2/lib-globals-scheduling.html
- https://lamport.azurewebsites.net/tla/high-level-view.html
- https://www.w3.org/TR/scxml/
- https://apalache-mc.org/docs/apalache/running.html#14-bounded-model-checking
- https://alloytools.org/spec.html

## Rollout, compatibility, and risks

The change is additive: existing family and rule IDs remain unchanged. The dynamically generated request schema gains three family enum members and their rule IDs. Clients that correctly consume the catalog remain compatible; clients that hard-code the old enum must update.

Primary risks and mitigations:

- **Scheduling blow-up:** enforce explicit horizons and generated-model budgets before solve.
- **False temporal claims:** include bound provenance in workflow outcomes and documentation.
- **Version ambiguity:** accept only caller-declared ordered ranks and ordering IDs.
- **Misleading optimization:** claim optimality only when the optimization envelope reports complete termination.
- **Attribution drift:** stable, deterministic constraint names include family, rule, and subject identity.
- **Manifest/compiler skew:** bundle validation fails closed on handler ownership or ABI mismatch.

No data migration is required. The implementation should land family by family behind tests, then expose all three through the catalog in the registration change.

## Formal proof evidence

All native formal cells were preflighted against their pinned registry versions and executed through the notebook solver. Every mandatory obligation matched and all freshness facets are current.

| Formal spec | Cell ID | Obligations | Source hash | IR hash | Report hash |
|---|---|---:|---|---|---|
| `GENERIC-FAMILY-ROUTING` | `e54e14d0-0000-4000-8000-000000000004` | 7/7 | `b59f6d1890ba06aa9d6f60476b38bbcd35e98553dad55c1695592dbd72919e0f` | `0ebdf6583792507689cff6fdc0031988c8c080e29e36cbc86155cb345c19dc63` | `4005619deda58986e8bac1e66c84ec41d6fe7463444052fbec8882c797d5d644` |
| `CONFIGURATION-FINITE-COMPATIBILITY` | `e54e14d0-0000-4000-8000-000000000006` | 5/5 | `c98c0b5859fb0e70ef52ead1278d6b20bb138ab3606b492f976e5b4856f2eccc` | `b858610e4d7f1fc9e9fe9c298143b5c7bfd95fc491fc128e5babe9441c3c4c22` | `6f24a2fce82ac48bd50d030ee5188f3b7abc7ffddcdd812b115a67bdf3e2ca51` |
| `SCHEDULING-FINITE-HORIZON-FEASIBILITY` | `e54e14d0-0000-4000-8000-000000000008` | 5/5 | `624128824dff665f5786dcec1f0cf172d6ae194b6eb12d52e93afdeb15ae7548` | `a643fdfe594e6260b944b5af580b25e9be452c716329cffa814f904f712951ce` | `add56c134eac965ed36b91d1fd95710daa01d51238c97ef97e37c921b7b6b0ea` |
| `WORKFLOW-APPROVAL-SAFETY` | `e54e14d0-0000-4000-8000-000000000010` | 8/8 | `9810568453d7f35f3f450fc2c2e0bc6bf58a62f92091125891ab14a6f48e358e` | `e1f446c091cbdf72d60949bace8cd057dbd72046a416f80927392e7c364dbe91` | `f2b08b91adbf3ede31d15ff695f359469b4e4d83e08c953c202cc8b6a4a1bc50` |

Persisted typed-solver results cited elsewhere in this notebook remain the authority for optimization and bounded counterexample claims that the live NS-Mermaid registry cannot express.